In [1]:
import os
!git clone https://github.com/uzbtrust/Uzbek-Operator-RAG-From-Scratch.git
os.chdir("Uzbek-Operator-RAG-From-Scratch")
!pip install -q tokenizers pyyaml

Cloning into 'Uzbek-Operator-RAG-From-Scratch'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 60 (delta 17), reused 58 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 75.31 KiB | 2.79 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
import shutil

os.makedirs("checkpoints/pretrain", exist_ok=True)
shutil.copy(
    "/kaggle/input/models/uzbtrust/uzbek-operator-rag-pretrained/pytorch/shard0-step130000/1/shard0_step130000.pt",
    "checkpoints/pretrain/shard0_step130000.pt"
)
print("checkpoint tayyor:", os.path.getsize("checkpoints/pretrain/shard0_step130000.pt") / 1e6, "MB")

checkpoint tayyor: 505.940027 MB


In [3]:
!python data/download_corpus.py --max-docs 50000
!python data/preprocess.py
!python tokenizer/train_tokenizer.py

2026-04-03 17:01:25,438 yangi shard: data/raw/shard_000.txt
2026-04-03 17:01:25,438 wikipedia yuklanmoqda...
2026-04-03 17:01:25,627 HTTP Request: HEAD https://huggingface.co/datasets/wikimedia/wikipedia/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-03 17:01:25,643 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/wikimedia/wikipedia/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/README.md "HTTP/1.1 200 OK"
2026-04-03 17:01:25,660 HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/wikimedia/wikipedia/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/README.md "HTTP/1.1 200 OK"
README.md: 131kB [00:00, 161MB/s]
2026-04-03 17:01:25,726 HTTP Request: HEAD https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/wikipedia.py "HTTP/1.1 404 Not Found"
2026-04-03 17:01:25,726 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads

In [4]:
!python data/synthetic_qa_generator.py
!python -c "import json; d=json.load(open('data/synthetic_qa.json')); print(f'jami: {len(d)} ta juftlik')"

2026-04-03 17:03:08,796 1015 ta QA juftlik yaratildi -> data/synthetic_qa.json
2026-04-03 17:03:08,796   address: 91
2026-04-03 17:03:08,796   complaints: 103
2026-04-03 17:03:08,796   contact: 88
2026-04-03 17:03:08,796   documents: 105
2026-04-03 17:03:08,796   no_info: 15
2026-04-03 17:03:08,796   payment: 110
2026-04-03 17:03:08,796   pricing: 102
2026-04-03 17:03:08,796   promotions: 94
2026-04-03 17:03:08,796   services: 101
2026-04-03 17:03:08,796   tech_support: 109
2026-04-03 17:03:08,796   working_hours: 97
jami: 1015 ta juftlik


In [5]:
!python training/finetune_simcse.py \
    --checkpoint checkpoints/pretrain/shard0_step130000.pt \
    --qa-data data/synthetic_qa.json

2026-04-03 17:03:11,821 encoder yuklandi: checkpoints/pretrain/shard0_step130000.pt
2026-04-03 17:03:12,461 device: cuda
2026-04-03 17:03:12,462 model parametrlari: 33.9M
2026-04-03 17:03:12,464 1000 ta QA juftlik yuklandi
2026-04-03 17:03:23,716 epoch 1/5 | loss: 3.9865 | acc: 0.0698 | vaqt: 5s
2026-04-03 17:03:23,918 eng yaxshi model saqlandi: checkpoints/simcse/best_model.pt (loss: 3.9865)
2026-04-03 17:03:26,987 epoch 2/5 | loss: 2.8907 | acc: 0.1646 | vaqt: 8s
2026-04-03 17:03:27,245 eng yaxshi model saqlandi: checkpoints/simcse/best_model.pt (loss: 2.8907)
2026-04-03 17:03:30,291 epoch 3/5 | loss: 2.2998 | acc: 0.1688 | vaqt: 11s
2026-04-03 17:03:30,553 eng yaxshi model saqlandi: checkpoints/simcse/best_model.pt (loss: 2.2998)
2026-04-03 17:03:33,716 epoch 4/5 | loss: 2.0929 | acc: 0.1792 | vaqt: 15s
2026-04-03 17:03:33,972 eng yaxshi model saqlandi: checkpoints/simcse/best_model.pt (loss: 2.0929)
2026-04-03 17:03:37,023 epoch 5/5 | loss: 2.0255 | acc: 0.1750 | vaqt: 18s
2026-04-

In [6]:
import os

best = "checkpoints/simcse/best_model.pt"
final = "checkpoints/simcse/final_model.pt"

if os.path.exists(best):
    print(f"best_model.pt: {os.path.getsize(best)/1e6:.1f} MB")
if os.path.exists(final):
    print(f"final_model.pt: {os.path.getsize(final)/1e6:.1f} MB")

best_model.pt: 135.8 MB
final_model.pt: 135.8 MB
